In [5]:
import random
import time
from datetime import datetime


class Agent:
    """
    A simulated agent that performs a fixed number of steps.
    This is a pure simulation: no LLMs, no APIs, no network calls.
    """

    def __init__(self, name, steps, fail_at_step=None):
        self.name = name
        self.steps = steps
        self.fail_at_step = fail_at_step  # optional deterministic failure step

    def run(self, listener):
        """
        Execute the agent step-by-step and emit structured progress events.
        """
        for step in range(1, self.steps + 1):
            # Simulate work / latency
            time.sleep(random.uniform(0.05, 0.2))

            # Simulate a failure if configured
            if self.fail_at_step is not None and step == self.fail_at_step:
                raise RuntimeError(f"{self.name} failed at step {step}")

            # Notify listener of progress
            listener(self.name, step, self.steps)


class Orchestrator:
    """
    Runs multiple agents sequentially and tracks workflow progress.
    Adds:
      - per-agent progress reporting
      - overall workflow progress
      - graceful failure handling
      - final summary
    """

    def __init__(self, agents, listener):
        self.agents = agents
        self.listener = listener

        # Precompute total workflow steps
        self.total_workflow_steps = sum(agent.steps for agent in agents)
        self.completed_workflow_steps = 0

        # Track status of each agent
        self.agent_status = {agent.name: "pending" for agent in agents}

    def _make_wrapped_listener(self):
        """
        Wrap the original listener so we can update overall workflow progress
        every time an agent reports a completed step.
        """
        def wrapped_listener(agent_name, step, total_steps):
            self.completed_workflow_steps += 1

            # Delegate to the user-provided listener
            self.listener(
                agent_name=agent_name,
                step=step,
                total_steps=total_steps,
                completed_workflow_steps=self.completed_workflow_steps,
                total_workflow_steps=self.total_workflow_steps
            )

        return wrapped_listener

    def run(self):
        """
        Run agents one by one. If an agent fails, stop the workflow gracefully
        and print a final summary.
        """
        wrapped_listener = self._make_wrapped_listener()

        print("=" * 70)
        print("Starting multi-agent workflow")
        print(f"Total agents: {len(self.agents)}")
        print(f"Total workflow steps: {self.total_workflow_steps}")
        print("=" * 70)

        start_time = time.time()

        for agent in self.agents:
            self.agent_status[agent.name] = "running"
            print(f"\n[START] {agent.name}")

            try:
                agent.run(wrapped_listener)
                self.agent_status[agent.name] = "completed"
                print(f"[DONE]  {agent.name}")

            except Exception as e:
                self.agent_status[agent.name] = "failed"
                print(f"[ERROR] {agent.name}: {e}")
                print("\nWorkflow stopped because an agent failed.")
                break

        end_time = time.time()
        duration = end_time - start_time

        self.print_summary(duration)

    def print_summary(self, duration):
        """
        Print a final workflow summary.
        """
        print("\n" + "=" * 70)
        print("WORKFLOW SUMMARY")
        print("=" * 70)

        for agent in self.agents:
            print(f"{agent.name:<12} -> {self.agent_status[agent.name]}")

        print("-" * 70)
        print(f"Completed workflow steps: {self.completed_workflow_steps}/{self.total_workflow_steps}")
        print(f"Total runtime: {duration:.2f} seconds")
        print("=" * 70)


def progress_listener(agent_name, step, total_steps, completed_workflow_steps, total_workflow_steps):
    """
    Structured progress listener.

    Prints:
      - current agent progress
      - agent-level percentage
      - overall workflow progress
      - timestamp
    """
    agent_pct = (step / total_steps) * 100
    workflow_pct = (completed_workflow_steps / total_workflow_steps) * 100
    timestamp = datetime.now().strftime("%H:%M:%S")

    print(
        f"[{timestamp}] "
        f"{agent_name}: step {step}/{total_steps} "
        f"({agent_pct:.0f}%) | "
        f"workflow {completed_workflow_steps}/{total_workflow_steps} "
        f"({workflow_pct:.0f}%)"
    )


def main():
    # Example workflow configuration
    agents = [
        Agent("Planner", 3),
        Agent("Researcher", 6),
        Agent("Writer", 4),
        Agent("Reviewer", 2),

        # Uncomment this to test failure handling:
        # Agent("Writer", 4, fail_at_step=2)
    ]

    orchestrator = Orchestrator(agents, progress_listener)
    orchestrator.run()


if __name__ == "__main__":
    main()

Starting multi-agent workflow
Total agents: 4
Total workflow steps: 15

[START] Planner
[06:08:49] Planner: step 1/3 (33%) | workflow 1/15 (7%)
[06:08:49] Planner: step 2/3 (67%) | workflow 2/15 (13%)
[06:08:49] Planner: step 3/3 (100%) | workflow 3/15 (20%)
[DONE]  Planner

[START] Researcher
[06:08:50] Researcher: step 1/6 (17%) | workflow 4/15 (27%)
[06:08:50] Researcher: step 2/6 (33%) | workflow 5/15 (33%)
[06:08:50] Researcher: step 3/6 (50%) | workflow 6/15 (40%)
[06:08:50] Researcher: step 4/6 (67%) | workflow 7/15 (47%)
[06:08:50] Researcher: step 5/6 (83%) | workflow 8/15 (53%)
[06:08:50] Researcher: step 6/6 (100%) | workflow 9/15 (60%)
[DONE]  Researcher

[START] Writer
[06:08:51] Writer: step 1/4 (25%) | workflow 10/15 (67%)
[06:08:51] Writer: step 2/4 (50%) | workflow 11/15 (73%)
[06:08:51] Writer: step 3/4 (75%) | workflow 12/15 (80%)
[06:08:51] Writer: step 4/4 (100%) | workflow 13/15 (87%)
[DONE]  Writer

[START] Reviewer
[06:08:51] Reviewer: step 1/2 (50%) | workflow 